# Instala las librerías necesarias para este notebook. 
- langchain_experimental permite usar PythonREPL, que ejecuta código Python dinámicamente.
- duckduckgo-search permite hacer búsquedas web usando DuckDuckGo.
-U actualiza la librería si ya existe.
-q reduce el ruido de salida durante la instalación.

In [14]:
!pip install ddgs

/home/ec2-user/anaconda3/envs/python3/lib/python3.12/pty.py:95: DeprecationWarning: This process (pid=8638) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


# Importa la librería json.
Sirve para convertir diccionarios/listas de Python a texto JSON y viceversa.

# Importa io.
Sirve para manejar flujos de entrada/salida en memoria.

# Importa display desde IPython.
Sirve para mostrar objetos visuales dentro de notebooks.

# Importa DDGS desde duckduckgo_search.
DDGS permite hacer búsquedas web usando DuckDuckGo.

# Importa pprint.
Sirve para imprimir estructuras complejas de forma más legible.

# Importa random.
Sirve para generar valores aleatorios si se necesitan.

# Importa boto3.
boto3 es el SDK oficial de AWS para Python.

# Importa sys.
Permite interactuar con variables y funciones del sistema Python.

# Importa StringIO.
Sirve para manejar texto como si fuera un archivo en memoria.

# Importa copy.
Sirve para copiar objetos sin modificar los originales.

# Importa PythonREPL desde langchain_experimental.
PythonREPL permite ejecutar código Python dinámicamente desde texto.

In [15]:
import json

import io

from IPython.display import display

#from duckduckgo_search import DDGS

from ddgs import DDGS

import pprint

import random

import boto3

import sys

from io import StringIO

import copy

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

from langchain_experimental.utilities import PythonREPL

# Define el ID del modelo que se usará en Amazon Bedrock.
En este caso se usa Claude 3 Sonnet de Anthropic.
# Define la región de AWS donde se invocará el modelo.
Aquí se usa us-west-2.
# Crea una sesión de boto3 usando la región indicada.
Esta sesión será usada para conectarse a servicios de AWS.
# Crea un cliente para Amazon Bedrock Runtime.
Este cliente permite invocar modelos usando la API Converse.

In [3]:
modelId = 'anthropic.claude-3-sonnet-20240229-v1:0'

region = 'us-west-2'

session = boto3.session.Session(region_name=region)

bedrock_client = session.client('bedrock-runtime')

# Define una función que será usada como herramienta del agente.
Esta herramienta recibe código Python y lo ejecuta.
# Crea una instancia de PythonREPL.
Esta instancia es como una consola de Python en la que se puede ejecutar código.
# Ejecuta el código Python pasado como string.
El resultado de la ejecución se guarda en result.
## Si ocurre cualquier error durante la ejecución del código...
Devuelve un mensaje indicando que falló la ejecución.
repr(e) muestra una representación técnica del error.
# Si el código se ejecutó correctamente, arma un mensaje de éxito.
Devuelve el mensaje final.

In [5]:
def chat_generator_from_python_code(code: str) -> str:
    """
    Function to executes the python code to generate the chart.
    Args:
        code: The python code that will generate the chart.
    """

    repl = PythonREPL()

    try:
        result = repl.run(code)

    except Exception as e:

        return f"Failed to execute. Error: {repr(e)}"

    result_str = f"Code has generated the chart successfully.\n{result}"

    return result_str

# Define una función que será usada como herramienta de búsqueda web.
Usa DuckDuckGo Search para buscar información.
keywords=query indica el texto que se buscará.
region='in-en' indica región India en inglés.
max_results=5 limita los resultados a 5.
Convierte cada resultado en texto JSON.
Luego une todos los resultados usando saltos de línea.
# Si ocurre un error durante la búsqueda...
Devuelve un mensaje indicando que la búsqueda falló.

In [16]:
def web_search(querye: str) -> str:
    """
    Function to research and collect more information to answer the query
    Args:
        query: The query that needs to be answered or more information needs to be collected.
    """

    try:

        results = DDGS().text(query=querye, region='in-en', max_results=5)

        return '\n'.join([json.dumps(result) for result in results])

    except Exception as e:

        return f"Failed to search. Error: {e}"

# Define una función utilitaria para llamar herramientas dinámicamente.
Busca dentro del espacio global una función con el nombre tool_name.
Por ejemplo, si tool_name = 'web_search',
entonces func será la función web_search.
Ejecuta la función pasando los parámetros como argumentos nombrados.
**parameters convierte un diccionario en argumentos.
## Ejemplo: 
{'query': 'India'} se convierte en query='India'.

In [7]:
def call_function(tool_name, parameters):

    func = globals()[tool_name]

    output = func(**parameters)

    return output

# Define una pregunta de ejemplo.
Llama la función web_search usando call_function.
tool_name es 'web_search'.
parameters contiene el parámetro query.

In [18]:
query = "What is the capital of India"

# Imprime la consulta que se enviará a la búsqueda web.
print(f"Query for Web search: \n{query}")

data = call_function('web_search', {'query': query})

# Imprime el resultado devuelto por la búsqueda.
print(f"Following is the output of web search: {data}")

Query for Web search: 
What is the capital of India
Following is the output of web search: {"title": "New Delhi - Wikipedia", "href": "https://en.wikipedia.org/wiki/New_Delhi", "body": "The national capital of India, New Delhi is jointly administered by both the Central Government of India and the local Government of Delhi, it is also the capital of the National Capital Territory (NCT) of Delhi."}
{"title": "What is the Capital of India? - WorldAtlas", "href": "https://www.worldatlas.com/articles/what-is-the-capital-of-india.html", "body": "Aug 9, 2018 \u00b7 New Delhi is located in the north-central part of India and is adjacent south of Delhi city. Initially, the capital city was in Kolkata when King George V of Britain ordered that the capital be moved to Delhi in 1911."}
{"title": "What is the Capital of India? - IndiaChakra", "href": "https://www.indiachakra.com/what-is-the-capital-of-india/", "body": "Feb 27, 2025 \u00b7 What is the Capital of India? The official capital of India

# Define la configuración de herramientas que recibirá el modelo.

In [10]:
toolConfig = {

    # Lista de herramientas disponibles.
    # Inicialmente está vacía.
    'tools': [],

    # Define cómo se elige la herramienta.
    'toolChoice': {

        # auto significa que el modelo decide automáticamente
        # si debe usar una herramienta o responder directamente.
        'auto': {}
    }
}

# Agrega una herramienta a la lista de herramientas disponibles.

In [11]:
toolConfig['tools'].append({

    # toolSpec describe una herramienta específica.
    'toolSpec': {

        # Nombre de la herramienta.
        # Debe coincidir con el nombre de la función Python.
        'name': 'web_search',

        # Descripción de la herramienta.
        # El modelo usa esta descripción para decidir cuándo llamarla.
        'description': 'Fetch information about any query from the internet.',

        # Define el esquema de entrada de la herramienta.
        'inputSchema': {
            'json': {

                # Indica que la entrada será un objeto JSON.
                'type': 'object',

                # Define las propiedades que acepta el objeto.
                'properties': {

                    # Define el parámetro query.
                    'query': {

                        # query debe ser un string.
                        'type': 'string',

                        # Describe qué representa query.
                        'description': 'Query for which more information is required.'
                    }
                },

                # Indica que query es obligatorio.
                'required': ['query']
            }
        }
    }
})

# Define una pregunta de ejemplo.

In [13]:
query = "What color was the white horse of Napoleon?"

# Imprime la consulta que se enviará a la búsqueda web.
print(f"Query for Web search: \n{query}")

# Llama la función web_search usando call_function.
# tool_name es 'web_search'.
# parameters contiene el parámetro query.
data = call_function('web_search', {'query': query})

# Imprime el resultado devuelto por la búsqueda.
print(f"Following is the output of web search: {data}")

Query for Web search: 
What color was the white horse of Napoleon?
Following is the output of web search: 


/tmp/ipykernel_8638/4098353327.py:10: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  results = DDGS().text(keywords=query, region='in-en', max_results=5)
